## Exploratory Data Analysis

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Load the data
column_names = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome"
]
df = pd.read_csv("../data/pima-indians-diabetes.csv", header=None, names = column_names)

print(df.info())
print("\n",df.describe())

In [ ]:
# Barplot (Diabetic count)

fig,ax = plt.subplots(figsize=(10,6))
sns.countplot(data = df, x = "Outcome", hue = "Outcome", ax = ax)
ax.set_title("Overall Diabetic count")
ax.legend(title = "Outcome", labels = ["Non-diabetic","diabetic"])
plt.tight_layout()
plt.show()

In [ ]:
# Percentage of diabetic and non-diabetic
print(df['Outcome'].value_counts(normalize = True)*100)

The data is imbalanced as the non diabetic indians are about 65% where as diabetic are about 35%.

In [ ]:
# Correlation heatmap
fig,ax = plt.subplots(figsize=(10,6))
corr = df.corr(numeric_only= True)
sns.heatmap(corr, annot = True, fmt = ".2f", cmap="coolwarm", ax = ax)
ax.set_title("Correlation Heatmap")

The Glucose(0.47) feature has the highest correlation with the Outcome.

## Data Preprocessing

In [ ]:
# Finding missing values
print(df.isna().sum())

# To count number of zeros in each column
(df==0).sum()


In [ ]:
# Finding the Missing Value
cols = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

df[cols] = df[cols].replace(0, np.nan)
print(df.isna().sum())

In [ ]:
# Handling the missing value
df=df.fillna(df.median(numeric_only=True))
print(df.isna().sum().sum())

In [ ]:
# Train/Test split
from sklearn.model_selection import train_test_split

# initalize the feature(X) and target(y)
X = df.drop(columns="Outcome")
y = df["Outcome"]

# Spliting data into Training(80%) and Test(20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training data: {X_train.shape}")
print(f"Testing data: {X_test.shape}")

In [ ]:
# Using Standard Scaler for scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Scaling the train and test data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Modeling

In [ ]:
## Logistic Regression Model
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    confusion_matrix
)

# Initializing the model
log_rg = LogisticRegression(max_iter=100, class_weight="balanced", random_state=42)

# Train the model
log_rg.fit(X_train_scaled, y_train)

# Prediction
y_pred = log_rg.predict(X_test_scaled)

# Evlaute
print(f"Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"Confusion matrix: \n")
print(confusion_matrix(y_test, y_pred))

In [ ]:
## XGBoost
from xgboost import XGBClassifier
import time

# Initializing scale_pos_weight
neg_count = (y_train==0).sum()
pos_count = (y_train==1).sum()
scale_pos_weight = neg_count/pos_count

# Initializing XGBoost Model
xgb = XGBClassifier(
    n_estimators = 100,
    learning_rate = 0.1,
    max_depth = 3,
    scale_pos_weight= scale_pos_weight,
    random_state = 42,
    eval_metric = "logloss"
)

# Training data
start_time = time.time()
xgb.fit(X_train, y_train) # xgboost doestn't require scaling
end_time = time.time()

print(f"Training time: {end_time - start_time:.2f}seconds.\n")

# Prediction
y_xgb_pred = xgb.predict(X_test)

# Evaluation
print(f"Accuracy: {accuracy_score(y_test, y_xgb_pred)*100:.2f}%")
print("Classification Report:")
print(classification_report(y_test, y_xgb_pred))
print(f"Confusion matrix: \n")
print(confusion_matrix(y_test, y_xgb_pred))


The XGBoost model has the better recall as well as accuracy for class 1(diabetic). Recall measures how many diabetic patients are actually identified. In the dataset like medical false negatives can be costly because an undiagoned patient may not receive the treatment. A model with high accuracy may still miss positive cases so, recall is often more meaningful and important metric.

## Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

# Create param_grid
param_grid = {
    'n_estimators': [50,100,200],
    'max_depth': [3,5,7]
}

# Setup Grid Search
grid_search = GridSearchCV(
    estimator=XGBClassifier(
        learning_rate =0.1,
        scale_pos_weight=scale_pos_weight,
        random_state =42,
        eval_metric='logloss'
    ),

    param_grid= param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
                           
)

# Run grid search
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}\n")
print(f"Best average cv score: {grid_search.best_score_*100:.2f}\n")

# Get best model
best_xgb_model = grid_search.best_estimator_
best_pred = best_xgb_model.predict(X_test)
print(f"Accuracy with best parameters: {accuracy_score(y_test, best_pred)*100:.2f}%")

Tuned model provides accuracy of 76.62% whereas baseline model has 75.97%

## Feature Importance

In [ ]:
# get important features from the best tuned xgb model
importances = best_xgb_model.feature_importances_

# create Dataframe for easyviewing
feature_imp_df= pd.DataFrame({
    'feature': X.columns,
    'Importance': importances

}).sort_values(by='Importance', ascending=False)

# barplot
fig,ax = plt.subplots(figsize=(10,6))
sns.barplot(data = feature_imp_df.head(3), x = 'Importance', y='feature', hue = 'feature', palette='Set2', ax=ax)
ax.set_title("Top 3 feature driving diabetes")
ax.set_xlabel("Relative Importance")
ax.set_ylabel("Features")
plt.tight_layout()
plt.show()

## Advice as a doctor to patient

Based on these findings, my primary advice would be to closely monitor and manage blood glucose levels, as glucose is the strongest predictor of diabetes risk. Regular blood sugar testing, limiting excessive sugar and refined carbohydrate intake, and following a balanced diet can help maintain healthy glucose levels.

BMI is the second most important factor, suggesting that body weight plays a significant role in diabetes risk. Maintaining a healthy weight through regular physical activity and proper nutrition can improve insulin sensitivity and reduce the likelihood of developing Type 2 Diabetes.

Age is another important risk factor. While age cannot be changed, individuals become more susceptible to diabetes as they grow older. Therefore, regular health checkups, routine diabetes screening, and proactive lifestyle management become increasingly important with age.

Overall, patients should focus on controlling blood sugar, maintaining a healthy body weight, staying physically active, and undergoing regular medical screenings, particularly as they get older.
